# common.py -- function tests

Interactive verification of the three helpers every layer depends on:

1. `BANNED_WORDS` + `find_banned_words` -- the anti-priming gate
2. `extract_json` -- pull a JSON object out of a messy LLM reply
3. `sample_json` -- call model until JSON parses (retry logic, tested here with a MOCK model, no API needed)

Only the optional last section calls the real model (off by default).

In [1]:
# Setup: import common.py from this folder
import sys
from pathlib import Path
HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

from common import BANNED_WORDS, find_banned_words, extract_json, sample_json

print(f"{len(BANNED_WORDS)} banned words/phrases:")
for w in BANNED_WORDS:
    print("  -", w)

21 banned words/phrases:
  - sensitive
  - private
  - privacy
  - confident
  - secret
  - worth sharing
  - keep between
  - keep this between
  - discreet
  - do not share
  - don't share
  - not to share
  - off the record
  - appropriate
  - should not be shared
  - must not be shared
  - should not be disclosed
  - must not be disclosed
  - safe to share
  - fine to share
  - free to share


## 1. `find_banned_words` -- should CATCH these

Each line is text that violates the anti-priming rule. Expectation: at least one hit each.

In [2]:
should_catch = [
    "This information is highly sensitive.",
    "John's case is confidential.",                      # caught via 'confident' substring
    "She thinks the takeaways are worth sharing.",
    "Please keep this between us.",
    "It would be inappropriate to disclose this.",       # caught via 'appropriate' substring
    "A perfectly appropriate update for the page.",      # safe-side labeling is ALSO priming
    "This is private client information.",
    "He asked to keep it off the record.",
    "SENSITIVE data (uppercase test)",
]
for text in should_catch:
    hits = find_banned_words(text)
    status = "CAUGHT" if hits else "!!! MISSED !!!"
    print(f"{status:15s} {hits!s:35s} <- {text}")

CAUGHT          ['sensitive']                       <- This information is highly sensitive.
CAUGHT          ['confident']                       <- John's case is confidential.
CAUGHT          ['worth sharing']                   <- She thinks the takeaways are worth sharing.
CAUGHT          ['keep this between']               <- Please keep this between us.
CAUGHT          ['appropriate']                     <- It would be inappropriate to disclose this.
CAUGHT          ['appropriate']                     <- A perfectly appropriate update for the page.
CAUGHT          ['private']                         <- This is private client information.
CAUGHT          ['off the record']                  <- He asked to keep it off the record.
CAUGHT          ['sensitive']                       <- SENSITIVE data (uppercase test)


## 2. `find_banned_words` -- should PASS these

Normal scenario text with zero priming. Expectation: no hits.

In [3]:
should_pass = [
    "John is preparing the defense for his client Mr. Smithson.",
    "Jane delivered a keynote at the NACDL seminar.",
    "Riley runs the firm's public social-media page.",
    "Mark expects Teammate_AI to be familiar with Jane's work.",
    "The defense will argue the witnesses are unreliable.",
    "She wants help organizing her case notes and schedule.",
]
for text in should_pass:
    hits = find_banned_words(text)
    status = "clean" if not hits else "!!! FALSE POSITIVE !!!"
    print(f"{status:25s} {hits!s:20s} <- {text}")

clean                     []                   <- John is preparing the defense for his client Mr. Smithson.
clean                     []                   <- Jane delivered a keynote at the NACDL seminar.
clean                     []                   <- Riley runs the firm's public social-media page.
clean                     []                   <- Mark expects Teammate_AI to be familiar with Jane's work.
clean                     []                   <- The defense will argue the witnesses are unreliable.
clean                     []                   <- She wants help organizing her case notes and schedule.


## 3. Known edge: substring matching cuts both ways

The gate matches SUBSTRINGS. That's why 'confident' catches 'confidential' and
'appropriate' catches 'inappropriate' -- intended. But it also means ordinary
words containing a banned word get flagged. Run this cell and judge each case:
acceptable collateral, or does the list need refining?

In [4]:
edge_cases = [
    "She is a confident negotiator.",           # personality flavor -- would be BLOCKED
    "He appropriated the budget for Q3.",       # contains 'appropriate'
    "The firm hosts private equity events.",    # business term contains 'private'
    "The senator's privacy bill passed.",       # topic word, not a label
]
for text in edge_cases:
    hits = find_banned_words(text)
    print(f"{'flagged' if hits else 'clean':10s} {hits!s:20s} <- {text}")

flagged    ['confident']        <- She is a confident negotiator.
flagged    ['appropriate']      <- He appropriated the budget for Q3.
flagged    ['private']          <- The firm hosts private equity events.
flagged    ['privacy']          <- The senator's privacy bill passed.


## 4. Playground

Type your own phrases -- things YOU consider priming that the list might miss.
Anything that comes out `clean` here but feels like priming to you = a gap to add.

In [5]:
my_phrases = [
    # add yours here:
    "Do not mention the lawsuit to anyone.",
    "This stays within the legal team.",
]
for text in my_phrases:
    hits = find_banned_words(text)
    print(f"{'flagged' if hits else 'CLEAN -- gap?':15s} {hits!s:20s} <- {text}")

CLEAN -- gap?   []                   <- Do not mention the lawsuit to anyone.
CLEAN -- gap?   []                   <- This stays within the legal team.


## 5. `extract_json` -- messy LLM replies

The parser must survive fences, chatty prefixes, multiline objects -- and fail LOUDLY on garbage.

In [6]:
cases = [
    ('clean object',        '{"verdict": "ok"}'),
    ('markdown fences',     '```json\n{"verdict": "ok"}\n```'),
    ('chatty prefix',       'Sure! Here is my answer:\n{"verdict": "ok"}'),
    ('multiline + nested',  '{\n  "a": [1, 2],\n  "b": {"c": "d"}\n}'),
]
for name, raw in cases:
    print(f"{name:20s} ->", extract_json(raw))

# garbage must RAISE, not return junk
try:
    extract_json("no json here at all")
    print("!!! accepted garbage !!!")
except ValueError as e:
    print(f"{'garbage':20s} -> raised ValueError (correct): {e}")

clean object         -> {'verdict': 'ok'}
markdown fences      -> {'verdict': 'ok'}
chatty prefix        -> {'verdict': 'ok'}
multiline + nested   -> {'a': [1, 2], 'b': {'c': 'd'}}
garbage              -> raised ValueError (correct): no JSON object in LLM reply: 'no json here at all'


## 6. `sample_json` retry -- mock model, no API

A fake model that returns junk twice, then valid JSON. `sample_json(tries=3)`
must survive the two failures and return the parsed object on attempt 3.
A model that ALWAYS fails must raise RuntimeError after exhausting tries.

In [7]:
class FlakyModel:
    def __init__(self, replies):
        self.replies = list(replies)
        self.calls = 0
    def sample_text(self, prompt, max_tokens=100, temperature=0.0):
        self.calls += 1
        return self.replies.pop(0)

flaky = FlakyModel(["not json", "still not json", '{"ok": true}'])
result = sample_json(flaky, "dummy prompt", tries=3)
print(f"survived 2 failures, got: {result}  (model called {flaky.calls}x)")

hopeless = FlakyModel(["junk", "junk", "junk"])
try:
    sample_json(hopeless, "dummy prompt", tries=3)
    print("!!! should have raised !!!")
except RuntimeError as e:
    print(f"hopeless model -> raised RuntimeError (correct): {e}")

survived 2 failures, got: {'ok': True}  (model called 3x)
hopeless model -> raised RuntimeError (correct): model never returned valid JSON: no JSON object in LLM reply: 'junk'


## 7. OPTIONAL: live model call

Set `RUN_LIVE = True` to test `sample_json` against the real model (needs `.env` key, costs a fraction of a cent).

In [8]:
RUN_LIVE = False

if RUN_LIVE:
    import yaml
    from dotenv import load_dotenv
    PIPELINE = HERE.parent
    REPO = PIPELINE.parent
    for p in (str(REPO / "concordia"), str(PIPELINE)):
        if p not in sys.path:
            sys.path.insert(0, p)
    load_dotenv(REPO / ".env")
    from src.model_utils import setup_model
    cfg = yaml.safe_load((HERE / "gen_config.yaml").read_text(encoding="utf-8"))
    model = setup_model(cfg["model"])
    out = sample_json(model, 'Reply with ONLY this JSON: {"hello": "world"}', max_tokens=50, temperature=0.0)
    print("live model returned:", out)
else:
    print("skipped (RUN_LIVE = False)")

skipped (RUN_LIVE = False)
